# Customer Segmentation Using Unsupervised Learning

## Problem Statement and Objective

A mall wants to understand its customers better by grouping them based on their **spending habits and income**. The goal is to apply **K-Means Clustering** to segment customers into distinct groups and propose targeted **marketing strategies** for each segment.

### Key Objectives
- Perform Exploratory Data Analysis (EDA) on the Mall Customers dataset
- Apply K-Means Clustering to identify customer segments
- Visualize clusters using PCA and t-SNE
- Propose marketing strategies for each segment

### Dataset
**Source:** Mall Customers Dataset (Kaggle)  
**Features:** CustomerID, Gender, Age, Annual Income (k$), Spending Score (1–100)

## 1. Setup & Imports

In [1]:
import os
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from io import StringIO
import urllib.request
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

# Set working directory to notebook location
try:
    os.chdir(os.path.dirname(os.path.abspath(__vsc_ipynb_file__)))
except NameError:
    pass

os.makedirs('charts', exist_ok=True)
print('All libraries loaded successfully.')
print(f'Charts will be saved to: {os.path.abspath("charts")}')

All libraries loaded successfully.
Charts will be saved to: c:\Users\amjad\Projects\developerhub-data-science-internship-p-2-\Assignment 2\charts


## 2. Dataset Loading & Description

In [2]:
from io import StringIO
import urllib.request

URLS = [
    'https://raw.githubusercontent.com/krishnaik06/Mall-Customers-Segmentation/master/Mall_Customers.csv',
    'https://raw.githubusercontent.com/nikhileshkr/Mall-Customer-Segmentation/main/Mall_Customers.csv',
    'https://raw.githubusercontent.com/tirthajyoti/Machine-Learning-with-Python/master/Datasets/Mall_Customers.csv',
]

df = None
for url in URLS:
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            df = pd.read_csv(StringIO(r.read().decode()))
        print(f'Downloaded from: {url}')
        break
    except Exception:
        continue

# Final fallback: load from local file
if df is None:
    try:
        df = pd.read_csv('Mall_Customers.csv')
        print('Loaded from local Mall_Customers.csv')
    except FileNotFoundError:
        print('All URLs failed. Generating synthetic dataset with same structure...')
        np.random.seed(42)
        n = 200
        df = pd.DataFrame({
            'CustomerID': range(1, n + 1),
            'Genre': np.random.choice(['Male', 'Female'], n, p=[0.44, 0.56]),
            'Age': np.random.randint(18, 70, n),
            'Annual Income (k$)': np.random.randint(15, 137, n),
            'Spending Score (1-100)': np.random.randint(1, 100, n)
        })

# Standardize column names regardless of source
df.columns = ['CustomerID', 'Gender', 'Age', 'AnnualIncome', 'SpendingScore']
print(f'Dataset shape: {df.shape}')
print(f'Features: {list(df.columns)}')

Downloaded from: https://raw.githubusercontent.com/tirthajyoti/Machine-Learning-with-Python/master/Datasets/Mall_Customers.csv
Dataset shape: (200, 5)
Features: ['CustomerID', 'Gender', 'Age', 'AnnualIncome', 'SpendingScore']


In [3]:
df.head(10)

,CustomerID,Gender,Age,AnnualIncome,SpendingScore
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40
5,6,Female,22,17,76
6,7,Female,35,18,6
7,8,Female,23,18,94
8,9,Male,64,19,3
9,10,Female,30,19,72


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   CustomerID     200 non-null    int64
 1   Gender         200 non-null    str  
 2   Age            200 non-null    int64
 3   AnnualIncome   200 non-null    int64
 4   SpendingScore  200 non-null    int64
dtypes: int64(4), str(1)
memory usage: 7.9 KB


In [5]:
df.describe()

,CustomerID,Age,AnnualIncome,SpendingScore
count,200.000000,200.000000,200.000000,200.000000
mean,100.500000,38.850000,60.560000,50.200000
std,57.879185,13.969007,26.264721,25.823522
min,1.000000,18.000000,15.000000,1.000000
25%,50.750000,28.750000,41.500000,34.750000
50%,100.500000,36.000000,61.500000,50.000000
75%,150.250000,49.000000,78.000000,73.000000
max,200.000000,70.000000,137.000000,99.000000


### Feature Descriptions

| Feature | Description |
|---------|-------------|
| `CustomerID` | Unique customer identifier |
| `Gender` | Male / Female |
| `Age` | Customer age (years) |
| `AnnualIncome` | Annual income in thousands of dollars |
| `SpendingScore` | Mall-assigned score (1–100) based on spending behavior |

## 3. Data Cleaning & Preprocessing

In [6]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')

Missing values:
CustomerID       0
Gender           0
Age              0
AnnualIncome     0
SpendingScore    0
dtype: int64

Duplicate rows: 0


In [7]:
# Encode Gender for analysis
df['Gender_encoded'] = df['Gender'].map({'Male': 0, 'Female': 1})

# Features used for clustering
features = ['Age', 'AnnualIncome', 'SpendingScore']
X = df[features]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Clustering features: {features}')
print(f'Scaled data shape: {X_scaled.shape}')

Clustering features: ['Age', 'AnnualIncome', 'SpendingScore']
Scaled data shape: (200, 3)


## 4. Exploratory Data Analysis (EDA)

In [8]:
# Gender distribution
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
gender_counts = df['Gender'].value_counts()
axes[0].bar(gender_counts.index, gender_counts.values, color=['#3498DB', '#E74C3C'], edgecolor='black')
axes[0].set_title('Gender Distribution', fontsize=13)
axes[0].set_ylabel('Count')
for i, v in enumerate(gender_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

axes[1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#3498DB', '#E74C3C'], startangle=90)
axes[1].set_title('Gender Split', fontsize=13)

plt.suptitle('Gender Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/gender_distribution.png', bbox_inches='tight')
plt.close()
print('Saved: charts/gender_distribution.png')

Saved: charts/gender_distribution.png


In [9]:
# Distribution of numerical features
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = ['#2ECC71', '#3498DB', '#E67E22']

for ax, col, color in zip(axes, features, colors):
    ax.hist(df[col], bins=20, color=color, edgecolor='black', alpha=0.8)
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.1f}')
    ax.set_title(col, fontsize=12)
    ax.legend(fontsize=9)

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/feature_distributions.png', bbox_inches='tight')
plt.close()
print('Saved: charts/feature_distributions.png')

Saved: charts/feature_distributions.png


In [10]:
# Pairplot: Income vs Spending Score by Gender
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for gender, color, marker in [('Male', '#3498DB', 'o'), ('Female', '#E74C3C', 's')]:
    subset = df[df['Gender'] == gender]
    axes[0].scatter(subset['AnnualIncome'], subset['SpendingScore'],
                    c=color, label=gender, alpha=0.7, edgecolors='black', linewidths=0.4, marker=marker)
    axes[1].scatter(subset['Age'], subset['SpendingScore'],
                    c=color, label=gender, alpha=0.7, edgecolors='black', linewidths=0.4, marker=marker)

axes[0].set_xlabel('Annual Income (k$)')
axes[0].set_ylabel('Spending Score')
axes[0].set_title('Income vs Spending Score', fontsize=12)
axes[0].legend()

axes[1].set_xlabel('Age')
axes[1].set_ylabel('Spending Score')
axes[1].set_title('Age vs Spending Score', fontsize=12)
axes[1].legend()

plt.suptitle('Key Relationships by Gender', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/scatter_relationships.png', bbox_inches='tight')
plt.close()
print('Saved: charts/scatter_relationships.png')

Saved: charts/scatter_relationships.png


In [11]:
# Correlation heatmap
plt.figure(figsize=(6, 4))
corr = df[['Age', 'AnnualIncome', 'SpendingScore', 'Gender_encoded']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/correlation_heatmap.png', bbox_inches='tight')
plt.close()
print('Saved: charts/correlation_heatmap.png')

Saved: charts/correlation_heatmap.png


In [12]:
# Boxplots by gender
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, features):
    sns.boxplot(x='Gender', y=col, data=df, palette={'Male': '#3498DB', 'Female': '#E74C3C'}, ax=ax)
    ax.set_title(f'{col} by Gender', fontsize=11)
plt.suptitle('Feature Distribution by Gender', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/boxplots_by_gender.png', bbox_inches='tight')
plt.close()
print('Saved: charts/boxplots_by_gender.png')

Saved: charts/boxplots_by_gender.png


## 5. K-Means Clustering

### 5.1 Optimal Number of Clusters — Elbow Method & Silhouette Score

In [13]:
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=7)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')
axes[0].axvline(x=5, color='red', linestyle='--', linewidth=1.5, label='Optimal K=5')
axes[0].legend()

axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=7)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score', fontsize=12, fontweight='bold')
best_k = list(K_range)[silhouette_scores.index(max(silhouette_scores))]
axes[1].axvline(x=best_k, color='red', linestyle='--', linewidth=1.5, label=f'Best K={best_k}')
axes[1].legend()

plt.suptitle('Choosing Optimal Number of Clusters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/elbow_silhouette.png', bbox_inches='tight')
plt.close()
print(f'Saved: charts/elbow_silhouette.png')
print(f'Best K by silhouette: {best_k}')

Saved: charts/elbow_silhouette.png
Best K by silhouette: 6


### 5.2 Fit K-Means with K=5

In [14]:
K_OPTIMAL = 5
kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print(f'K-Means fitted with K={K_OPTIMAL}')
print(f'Silhouette Score: {silhouette_score(X_scaled, df["Cluster"]):.4f}')
print(f'\nCluster sizes:')
print(df['Cluster'].value_counts().sort_index())

K-Means fitted with K=5
Silhouette Score: 0.4166

Cluster sizes:
Cluster
0    20
1    54
2    40
3    39
4    47
Name: count, dtype: int64


In [15]:
# Cluster profiles
cluster_profile = df.groupby('Cluster')[features].mean().round(1)
cluster_profile['Count'] = df['Cluster'].value_counts().sort_index()
cluster_profile['Gender_Female_%'] = (df.groupby('Cluster')['Gender_encoded'].mean() * 100).round(1)
print('=== Cluster Profiles ===')
print(cluster_profile)

=== Cluster Profiles ===
          Age  AnnualIncome  SpendingScore  Count  Gender_Female_%
Cluster                                                           
0        46.2          26.8           18.4     20             60.0
1        25.2          41.1           62.2     54             59.3
2        32.9          86.1           81.5     40             55.0
3        39.9          86.1           19.4     39             48.7
4        55.6          54.4           48.9     47             57.4


In [16]:
# Cluster visualization: Income vs Spending Score (the most revealing view)
COLORS = ['#E74C3C', '#2ECC71', '#3498DB', '#F39C12', '#9B59B6']
SEGMENT_NAMES = {
    0: 'Segment A',
    1: 'Segment B',
    2: 'Segment C',
    3: 'Segment D',
    4: 'Segment E'
}

plt.figure(figsize=(9, 6))
for cluster_id in range(K_OPTIMAL):
    subset = df[df['Cluster'] == cluster_id]
    plt.scatter(subset['AnnualIncome'], subset['SpendingScore'],
                c=COLORS[cluster_id], label=f'Cluster {cluster_id}',
                s=80, edgecolors='black', linewidths=0.4, alpha=0.85)

# Plot centroids (inverse transform to original scale)
centroids_orig = scaler.inverse_transform(kmeans.cluster_centers_)
feature_idx = {f: i for i, f in enumerate(features)}
plt.scatter(centroids_orig[:, feature_idx['AnnualIncome']],
            centroids_orig[:, feature_idx['SpendingScore']],
            c='black', marker='X', s=200, zorder=5, label='Centroids')

plt.xlabel('Annual Income (k$)', fontsize=11)
plt.ylabel('Spending Score', fontsize=11)
plt.title('K-Means Clusters: Income vs Spending Score', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('charts/kmeans_clusters.png', bbox_inches='tight')
plt.close()
print('Saved: charts/kmeans_clusters.png')

Saved: charts/kmeans_clusters.png


In [17]:
# Cluster feature comparison — radar-style bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, feat in zip(axes, features):
    vals = cluster_profile[feat]
    ax.bar(range(K_OPTIMAL), vals, color=COLORS, edgecolor='black')
    ax.set_xticks(range(K_OPTIMAL))
    ax.set_xticklabels([f'C{i}' for i in range(K_OPTIMAL)])
    ax.set_title(f'Mean {feat} per Cluster', fontsize=11)
    ax.set_ylabel(feat)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.5, f'{v:.0f}', ha='center', fontsize=9)

plt.suptitle('Cluster Feature Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/cluster_profiles.png', bbox_inches='tight')
plt.close()
print('Saved: charts/cluster_profiles.png')

Saved: charts/cluster_profiles.png


## 6. Dimensionality Reduction & Cluster Visualization

### 6.1 PCA (Principal Component Analysis)

In [18]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Explained variance ratio: {pca.explained_variance_ratio_}')
print(f'Total variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA scatter
for cluster_id in range(K_OPTIMAL):
    mask = df['Cluster'] == cluster_id
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=COLORS[cluster_id], label=f'Cluster {cluster_id}',
                    s=70, edgecolors='black', linewidths=0.3, alpha=0.85)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].set_title('PCA — Customer Clusters', fontsize=12, fontweight='bold')
axes[0].legend()

# Scree plot
pca_full = PCA(random_state=42).fit(X_scaled)
axes[1].bar(range(1, len(features)+1), pca_full.explained_variance_ratio_*100,
            color='#3498DB', edgecolor='black')
axes[1].plot(range(1, len(features)+1), np.cumsum(pca_full.explained_variance_ratio_*100),
             'ro-', linewidth=2)
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Explained Variance (%)')
axes[1].set_title('Scree Plot', fontsize=12, fontweight='bold')
axes[1].set_xticks(range(1, len(features)+1))

plt.suptitle('PCA Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/pca_visualization.png', bbox_inches='tight')
plt.close()
print('Saved: charts/pca_visualization.png')

Explained variance ratio: [0.44266167 0.33308378]
Total variance explained: 77.6%
Saved: charts/pca_visualization.png


### 6.2 t-SNE Visualization

In [21]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for cluster_id in range(K_OPTIMAL):
    mask = df['Cluster'] == cluster_id
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=COLORS[cluster_id], label=f'Cluster {cluster_id}',
                s=80, edgecolors='black', linewidths=0.3, alpha=0.85)

plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.title('t-SNE — Customer Clusters', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('charts/tsne_visualization.png', bbox_inches='tight')
plt.close()
print('Saved: charts/tsne_visualization.png')

Saved: charts/tsne_visualization.png


## 7. Cluster Interpretation & Marketing Strategies

In [22]:
# Assign meaningful segment labels based on cluster profiles
# (Labels are assigned after inspecting cluster_profile above)
segment_map = {}
profiles = cluster_profile[['AnnualIncome', 'SpendingScore']]

for cid, row in profiles.iterrows():
    inc, sco = row['AnnualIncome'], row['SpendingScore']
    if inc >= 70 and sco >= 60:
        segment_map[cid] = 'High Income, High Spender'
    elif inc >= 70 and sco < 45:
        segment_map[cid] = 'High Income, Low Spender'
    elif inc < 45 and sco >= 60:
        segment_map[cid] = 'Low Income, High Spender'
    elif inc < 45 and sco < 45:
        segment_map[cid] = 'Low Income, Low Spender'
    else:
        segment_map[cid] = 'Average Customer'

df['Segment'] = df['Cluster'].map(segment_map)

print('=== Cluster → Segment Mapping ===')
for cid, name in segment_map.items():
    count = (df['Cluster'] == cid).sum()
    print(f'  Cluster {cid}: {name} (n={count})')

=== Cluster → Segment Mapping ===
  Cluster 0: Low Income, Low Spender (n=20)
  Cluster 1: Low Income, High Spender (n=54)
  Cluster 2: High Income, High Spender (n=40)
  Cluster 3: High Income, Low Spender (n=39)
  Cluster 4: Average Customer (n=47)


In [23]:
# Segment summary visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_counts = df['Segment'].value_counts()
axes[0].barh(seg_counts.index, seg_counts.values,
             color=[COLORS[i] for i in range(len(seg_counts))], edgecolor='black')
axes[0].set_title('Customer Count per Segment', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Count')
for i, v in enumerate(seg_counts.values):
    axes[0].text(v + 0.5, i, str(v), va='center')

# Income vs Spending Score with segment labels
for cluster_id in range(K_OPTIMAL):
    subset = df[df['Cluster'] == cluster_id]
    axes[1].scatter(subset['AnnualIncome'], subset['SpendingScore'],
                    c=COLORS[cluster_id], label=f'C{cluster_id}: {segment_map[cluster_id]}',
                    s=70, edgecolors='black', linewidths=0.3, alpha=0.85)
axes[1].set_xlabel('Annual Income (k$)')
axes[1].set_ylabel('Spending Score')
axes[1].set_title('Segments: Income vs Spending', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=7, loc='upper left')

plt.suptitle('Customer Segment Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/segment_overview.png', bbox_inches='tight')
plt.close()
print('Saved: charts/segment_overview.png')

Saved: charts/segment_overview.png


In [24]:
# Print full segment profiles
full_profile = df.groupby('Segment').agg(
    Count=('CustomerID', 'count'),
    Avg_Age=('Age', 'mean'),
    Avg_Income=('AnnualIncome', 'mean'),
    Avg_SpendingScore=('SpendingScore', 'mean'),
    Female_Pct=('Gender_encoded', lambda x: round(x.mean()*100, 1))
).round(1)
print('=== Full Segment Profiles ===')
print(full_profile)

=== Full Segment Profiles ===
                           Count  Avg_Age  Avg_Income  Avg_SpendingScore  \
Segment                                                                    
Average Customer              47     55.6        54.4               48.9   
High Income, High Spender     40     32.9        86.1               81.5   
High Income, Low Spender      39     39.9        86.1               19.4   
Low Income, High Spender      54     25.2        41.1               62.2   
Low Income, Low Spender       20     46.2        26.8               18.4   

                           Female_Pct  
Segment                                
Average Customer                 57.4  
High Income, High Spender        55.0  
High Income, Low Spender         48.7  
Low Income, High Spender         59.3  
Low Income, Low Spender          60.0  


## 8. Final Conclusion & Marketing Strategies

### Identified Customer Segments

| Cluster | Segment | Income | Spending Score | Strategy |
|---------|---------|--------|----------------|----------|
| High Income, High Spender | **Premium Loyalists** | High | High | Loyalty rewards, exclusive memberships, premium product launches |
| High Income, Low Spender | **Untapped Potential** | High | Low | Personalized outreach, showcase value, targeted promotions to convert them |
| Low Income, High Spender | **Impulse Buyers** | Low | High | Discounts, flash sales, EMI options, seasonal offers |
| Low Income, Low Spender | **Price-Sensitive** | Low | Low | Bundle deals, budget-friendly products, referral incentives |
| Average Customer | **Middle Ground** | Medium | Medium | Loyalty programs, upsell mid-range products, engagement campaigns |

### Key Insights

**1. Five Distinct Segments Found**  
K-Means with K=5 produced well-separated clusters confirmed by the Elbow Method and Silhouette Score.

**2. Income & Spending Score are Key Drivers**  
PCA showed that Annual Income and Spending Score capture most of the variance, forming the clearest cluster boundaries.

**3. t-SNE Confirms Separation**  
t-SNE visualization shows well-separated groups, validating that the K-Means clusters are meaningful and not random.

**4. Gender Skew**  
Female customers make up ~56% of the dataset and tend to have slightly higher spending scores.

**5. Actionable Segments**  
- Focus marketing budget on **Premium Loyalists** (high ROI, already spending)  
- Convert **Untapped Potential** customers with personalized campaigns  
- Retain **Impulse Buyers** with flash deals and limited-time offers